#  Setpoint GNN — Training & Evaluation

---

## Objective

Train a **GATv2-based Graph Neural Network** that predicts decentralized setpoint
commands `[Δx, Δy, Δz, Δyaw]` for a drone swarm, cloning an expert PID controller
via imitation learning.

### Model → Navigator Level
```
 Strategy GNN (friend)  →  Navigator GNN (this model)  →  PID Pilot (PyFlyt)
 "who goes where"           "how to get there"             "motor physics"
```

### Architecture: SetpointGATv2
```
 x [N, 10] ──►  GATv2 Layer 1 (10→64, 4 heads) ──►  hop 1: comm radius 10m
                        │ + skip connection
                        ▼
                 GATv2 Layer 2 (64→64, 4 heads)  ──►  hop 2: 2-neighbors deep
                        │ + skip connection
                        ▼
                 GATv2 Layer 3 (64→64, 4 heads)  ──►  hop 3: 3-neighbors deep
                        │
                        ▼
                 MLP Head: Linear(64→32) → ReLU → Linear(32→4)
                        │
                        ▼
               pred [N, 4]  (normalized setpoints, NO Sigmoid)
```

### Notebook Sections
1. Setup & Data Loading
2. Model Architecture (SetpointGATv2)
3. Training Stack (Optimizer, Scheduler, Grad Clip)
4. Training & Validation Loop
5. Evaluation & Physical Metrics
6. DAgger Framework

---

**Reference:** arXiv:1903.10527 — *Learning Decentralized Controllers for Robot Swarms with GNNs*

---
## 1. Setup & Data Loading

In [1]:
import json
import sys
import time
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.loader import DataLoader

# Paths
NOTEBOOK_DIR  = Path().resolve()
PROJECT_ROOT  = NOTEBOOK_DIR.parents[2]
SRC_DIR       = NOTEBOOK_DIR.parent / "src"
RESULTS_DIR   = NOTEBOOK_DIR.parent / "results"
CHECKPOINTS   = NOTEBOOK_DIR.parent / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
DATASETS_DIR  = PROJECT_ROOT / "datasets"

# Add src to path so we can import dataset.py
sys.path.insert(0, str(SRC_DIR))
from dataset import (
    DroneSwarmDataset,
    SetpointNormalizer,
    NormalizedDroneDataset,
    get_dataloaders,
    INPUT_NAMES, LABEL_NAMES, EDGE_NAMES,
)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\aimen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load data using our dataset.py module 
BATCH_SIZE = 32

train_loader, val_loader, test_loader, normalizer = get_dataloaders(
    dataset_dir=DATASETS_DIR,
    dataset_name="setpoint_mixed_v1_mixed_formations",
    batch_size=BATCH_SIZE,
    num_workers=0,  # 0 on Windows; increase on Linux/Colab
)

# Quick shape check
batch = next(iter(train_loader))
print(f"\nData loaded ")
print(f"  Train batches  : {len(train_loader)}")
print(f"  Val batches    : {len(val_loader)}")
print(f"  Test batches   : {len(test_loader)}")
print(f"\n  batch.x        : {tuple(batch.x.shape)}  (input: {len(INPUT_NAMES)}-dim)")
print(f"  batch.y        : {tuple(batch.y.shape)}  (label: {len(LABEL_NAMES)}-dim)")
print(f"  batch.edge_attr: {tuple(batch.edge_attr.shape)}  (edges: {len(EDGE_NAMES)}-dim)")
print(f"  batch.batch    : {tuple(batch.batch.shape)}  ({batch.num_graphs} graphs)")

FileNotFoundError: Dataset file not found: C:\Users\aimen\OneDrive\Desktop\gnn_drone_project\datasets\setpoint_mixed_v1_mixed_formations_train.pt

---
## 2. Model Architecture — SetpointGATv2

### Design Rationale

| Decision | Choice | Reason |
|----------|--------|--------|
| **Conv type** | GATv2Conv | Dynamic attention; learns to weight neighbors by relevance |
| **K = 3 layers** | 3-hop message passing | Covers 30m radius (3×10m) — enough for 10-20 drone swarms |
| **Heads = 4** | Multi-head attention | Different heads specialize (distance, direction, formation) |
| **Hidden = 64** | Per-head dim = 16 | 4 heads × 16 = 64 total — good capacity without overfitting |
| **Skip connections** | Residual after each layer | Preserves local features through deep message passing |
| **edge_dim = 4** | `[rel_x, rel_y, rel_z, distance]` | Edge-conditioned attention (spatial awareness) |
| **Dropout = 0.05** | Very light | Clean data — heavy dropout would discard useful signal |
| **Final activation** | None (Linear) | Setpoints are unbounded real values |

In [ ]:
class SetpointGATv2(nn.Module):
    """
    GATv2-based Navigator GNN for decentralized setpoint prediction.

    Architecture:
      3 × GATv2Conv (K=3 hop message passing, 4 attention heads)
      + residual (skip) connections after each GATv2 layer
      + 2-layer MLP head → 4-dim setpoint output (unbounded)

    Forward signature matches PyG convention:
      pred = model(x, edge_index, edge_attr)

    Args:
        in_channels:  Input feature dim per node (default: 10)
        edge_dim:     Edge feature dim (default: 4)
        hidden_dim:   Hidden dimension after each GATv2 layer (default: 64)
        out_channels: Output dim — setpoint components (default: 4)
        heads:        Number of attention heads per layer (default: 4)
        num_layers:   Number of GATv2 layers / K-hops (default: 3)
        dropout:      Dropout rate (default: 0.05)
    """

    def __init__(
        self,
        in_channels:  int = 10,
        edge_dim:     int = 4,
        hidden_dim:   int = 64,
        out_channels: int = 4,
        heads:        int = 4,
        num_layers:   int = 3,
        dropout:      float = 0.05,
    ):
        super().__init__()

        self.num_layers = num_layers
        self.dropout    = dropout

        # Store hyperparams for checkpointing
        self.hparams = {
            "in_channels":  in_channels,
            "edge_dim":     edge_dim,
            "hidden_dim":   hidden_dim,
            "out_channels": out_channels,
            "heads":        heads,
            "num_layers":   num_layers,
            "dropout":      dropout,
        }

        # ── GATv2 Layers ──────────────────────────────────────────────────────
        self.convs   = nn.ModuleList()
        self.norms   = nn.ModuleList()   # LayerNorm for stable training
        self.skips   = nn.ModuleList()   # Linear projections for skip connections

        for i in range(num_layers):
            c_in = in_channels if i == 0 else hidden_dim
            out_per_head = hidden_dim // heads

            self.convs.append(
                GATv2Conv(
                    in_channels  = c_in,
                    out_channels = out_per_head,
                    heads        = heads,
                    edge_dim     = edge_dim,
                    concat       = True,
                    dropout      = dropout,
                    add_self_loops = True,
                )
            )
            self.norms.append(nn.LayerNorm(hidden_dim))

            if c_in != hidden_dim:
                self.skips.append(nn.Linear(c_in, hidden_dim, bias=False))
            else:
                self.skips.append(nn.Identity())

        # ── MLP Head ──────────────────────────────────────────────────────────
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, out_channels),
        )

    def forward(self, x, edge_index, edge_attr):
        """
        Forward pass.

        Args:
            x:          [N, 10]   Node features (normalized state with sin/cos yaw)
            edge_index: [2, E]   Communication graph
            edge_attr:  [E, 4]   Edge features (normalized rel_xyz + distance)

        Returns:
            pred:       [N, 4]   Predicted setpoint error (normalized, local frame)
        """
        h = x
        for i in range(self.num_layers):
            h_new = self.convs[i](h, edge_index, edge_attr)
            h_skip = self.skips[i](h)
            h_new = h_new + h_skip
            h_new = self.norms[i](h_new)
            h_new = F.elu(h_new)
            h_new = F.dropout(h_new, p=self.dropout, training=self.training)
            h = h_new

        return self.mlp(h)


# ── Instantiate ────────────────────────────────────────────────────────────────
model = SetpointGATv2(
    in_channels  = 10, # Correctly set to 10
    edge_dim     = normalizer.to_dict()["edge_dim"],
    hidden_dim   = 64,
    out_channels = normalizer.to_dict()["output_dim"],
    heads        = 4,
    num_layers   = 3,
    dropout      = 0.05,
).to(DEVICE)

# ── Parameter count ────────────────────────────────────────────────────────────
total_params   = sum(p.numel() for p in model.parameters())
trainable      = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: SetpointGATv2")
print(f"  Layers       : {model.num_layers} × GATv2Conv (K=3 hops)")
print(f"  Heads        : {model.hparams['heads']}")
print(f"  Hidden dim   : {model.hparams['hidden_dim']}")
print(f"  Parameters   : {total_params:,} ({trainable:,} trainable)")
print(f"  Input        : {model.hparams['in_channels']}-dim node, {model.hparams['edge_dim']}-dim edge")
print(f"  Output       : {model.hparams['out_channels']}-dim setpoint (local, unbounded)")
print(f"  Device       : {DEVICE}")

# ── Quick smoke test ───────────────────────────────────────────────────────────
batch = next(iter(train_loader))
batch = batch.to(DEVICE)
with torch.no_grad():
    out = model(batch.x, batch.edge_index, batch.edge_attr)
print(f"\nSmoke test ✅")
print(f"  Input:  {tuple(batch.x.shape)}")
print(f"  Output: {tuple(out.shape)}")


Model: SetpointGATv2
  Layers       : 3 × GATv2Conv (K=3 hops)
  Heads        : 4
  Hidden dim   : 64
  Parameters   : 22,244 (22,244 trainable)
  Input        : 9-dim node, 4-dim edge
  Output       : 4-dim setpoint (unbounded)
  Device       : cuda

Smoke test ✅
  Input:  (440, 9)
  Output: (440, 4)


---
## 3. Training Stack

| Component | Choice | Reason |
|-----------|--------|--------|
| **Optimizer** | Adam (lr=1e-4, wd=1e-6) | Standard for GNNs; light L2 reg |
| **Scheduler** | ReduceLROnPlateau | Auto-tunes LR when val loss stalls |
| **Grad clip** | max_norm=1.0 | Prevents gradient explosion from outlier graphs |
| **Loss** | MSELoss | Regression in normalized space |
| **Early stop** | Patience=15 | Stops if val loss doesn't improve |

In [7]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 1e-6     # L2 regularization
GRAD_CLIP_NORM = 1.0      # max gradient norm
EPOCHS         = 100
PATIENCE       = 15       # early stopping patience
SCHEDULER_PATIENCE = 5    # LR reduction patience
SCHEDULER_FACTOR   = 0.5  # LR *= 0.5 on plateau

# ── Optimizer ──────────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# ── Scheduler ──────────────────────────────────────────────────────────────────
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
)

# ── Loss ────────────────────────────────────────────────────────────────────────
criterion = nn.MSELoss()

print(f"Optimizer : Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler : ReduceLROnPlateau (patience={SCHEDULER_PATIENCE}, factor={SCHEDULER_FACTOR})")
print(f"Grad clip : {GRAD_CLIP_NORM}")
print(f"Epochs    : {EPOCHS} (early stop patience={PATIENCE})")
print(f"Loss      : MSELoss (normalized space)")

Optimizer : Adam (lr=0.0001, wd=1e-06)
Scheduler : ReduceLROnPlateau (patience=5, factor=0.5)
Grad clip : 1.0
Epochs    : 100 (early stop patience=15)
Loss      : MSELoss (normalized space)


In [8]:
# Checkpointing 

def save_gnn_model(
    model: SetpointGATv2,
    normalizer: SetpointNormalizer,
    optimizer,
    epoch: int,
    train_loss: float,
    val_loss: float,
    path: Path,
):
    """
    Save a complete checkpoint: model weights + hyperparams + normalizer.

    Everything needed to resume training OR deploy at inference is in one file.
    """
    checkpoint = {
        # Model
        "model_state_dict":     model.state_dict(),
        "model_hparams":        model.hparams,

        # Optimizer (for resuming training)
        "optimizer_state_dict": optimizer.state_dict(),

        # Training state
        "epoch":       epoch,
        "train_loss":  train_loss,
        "val_loss":    val_loss,

        # Normalizer (needed at inference to denormalize predictions)
        "normalizer":  normalizer.to_dict(),
    }
    torch.save(checkpoint, path)
    print(f"  💾 Checkpoint saved → {path.name} (val_loss={val_loss:.6f})")


def load_gnn_model(path: Path, device="cpu"):
    """
    Load a complete checkpoint.

    Returns:
        (model, normalizer, checkpoint_dict)
    """
    ckpt = torch.load(path, map_location=device, weights_only=False)

    model = SetpointGATv2(**ckpt["model_hparams"]).to(device)
    model.load_state_dict(ckpt["model_state_dict"])

    normalizer = SetpointNormalizer.from_dict(ckpt["normalizer"])

    return model, normalizer, ckpt


print("Checkpoint utilities ready ✅")

Checkpoint utilities ready ✅


---
## 4. Training & Validation Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, grad_clip, noise_std=0.0):
    """
    One training epoch. Returns average MSE loss.
    Applies Gaussian noise to input features if noise_std > 0.
    """
    model.train()
    total_loss = 0.0
    total_nodes = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # --- Add Training Noise ---
        if noise_std > 0:
            batch.x = batch.x + torch.randn_like(batch.x) * noise_std

        pred = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(pred, batch.y)

        loss.backward()

        # Gradient clipping — prevents explosion from outlier graphs
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        n = batch.x.shape[0]  # number of nodes
        total_loss += loss.item() * n
        total_nodes += n

    return total_loss / total_nodes


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Evaluate on a split. Returns average MSE loss.
    """
    model.eval()
    total_loss = 0.0
    total_nodes = 0

    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(pred, batch.y)

        n = batch.x.shape[0]
        total_loss += loss.item() * n
        total_nodes += n

    return total_loss / total_nodes


print("Training utilities ready ✅")


Training utilities ready ✅


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MAIN TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

best_val_loss    = float("inf")
best_epoch       = 0
epochs_no_improve = 0
history          = {"train_loss": [], "val_loss": [], "lr": []}

BEST_MODEL_PATH = CHECKPOINTS / "best_setpoint_gatv2.pt"

# --- Training Noise ---
NOISE_STD = 0.01

print("=" * 75)
print(f"{'Epoch':>6}  {'Train Loss':>12}  {'Val Loss':>12}  {'LR':>10}  {'Time':>6}  Status")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ──────────────────────────────────────────────────────────────────
    # Apply noise only during training
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE, GRAD_CLIP_NORM,
        noise_std=NOISE_STD
    )

    # ── Validate ───────────────────────────────────────────────────────────────
    # No noise for validation
    val_loss = evaluate(model, val_loader, criterion, DEVICE)

    # ── Scheduler step ─────────────────────────────────────────────────────────
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    # ── Record history ─────────────────────────────────────────────────────────
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)

    elapsed = time.time() - t0

    # ── Checkpointing ──────────────────────────────────────────────────────────
    status = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_no_improve = 0
        status = "⬇ best"
        save_gnn_model(
            model, normalizer, optimizer, epoch,
            train_loss, val_loss, BEST_MODEL_PATH
        )
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            status = "🛑 early stop"

    print(f"  {epoch:>4}  {train_loss:>12.6f}  {val_loss:>12.6f}  {current_lr:>10.2e}  {elapsed:>5.1f}s  {status}")

    # ── Early stopping ─────────────────────────────────────────────────────────
    if epochs_no_improve >= PATIENCE:
        print(f"\n🛑 Early stopping at epoch {epoch}. Best was epoch {best_epoch}.")
        break

print(f"\n{'='*75}")
print(f"Training complete. Best val loss: {best_val_loss:.6f} at epoch {best_epoch}")
print(f"Checkpoint: {BEST_MODEL_PATH}")

In [ ]:
# ── Loss curves ────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
epochs_range = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs_range, history['train_loss'], label='Train', color='#4a90d9', lw=2)
ax1.plot(epochs_range, history['val_loss'], label='Validation', color='#e74c3c', lw=2)
ax1.axvline(best_epoch, color='#27ae60', ls='--', alpha=0.7, label=f'Best (ep {best_epoch})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss (normalized)')
ax1.set_title('Training & Validation Loss', fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_yscale('log')

# Learning rate
ax2.plot(epochs_range, history['lr'], color='#8e44ad', lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule', fontweight='bold')
ax2.set_yscale('log'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150)
plt.show()

---
## 5. Evaluation & Physical Metrics

All metrics are computed in **physical units** (metres, radians) after
denormalizing the model's predictions using the `SetpointNormalizer`.

### Success Criteria

| Metric | Target | Meaning |
|--------|--------|---------|
| MAE (position) | < 0.5 m | Average error in Δx, Δy, Δz |
| MAE (yaw) | < 0.1 rad | Average heading error |
| Euclidean 3D error | < 0.5 m | 90th percentile formation slot error |
| Scalability | Works on 100 drones | Zero-shot generalization beyond training |

In [ ]:
# ── Load best model ────────────────────────────────────────────────────────────
best_model, best_normalizer, ckpt = load_gnn_model(BEST_MODEL_PATH, device=DEVICE)
best_model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.6f})")

In [ ]:
# ── Collect predictions on TEST set ────────────────────────────────────────────
all_pred_norm  = []
all_label_norm = []
all_original_yaw = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        pred = best_model(batch.x, batch.edge_index, batch.edge_attr)
        all_pred_norm.append(pred.cpu())
        all_label_norm.append(batch.y.cpu())
        all_original_yaw.append(batch.original_yaw.cpu())

pred_norm  = torch.cat(all_pred_norm,  dim=0)
label_norm = torch.cat(all_label_norm, dim=0)
original_yaw = torch.cat(all_original_yaw, dim=0)

# ── Denormalize to physical units (and convert back to global frame) ───────────
pred_real_global = best_normalizer.denormalize_target(pred_norm, original_yaw=original_yaw)

# To get the ground truth in the global frame, we must also apply the inverse transform
# to the normalized labels.
label_real_global = best_normalizer.denormalize_target(label_norm, original_yaw=original_yaw)


print(f"Test set: {pred_real_global.shape[0]:,} node predictions")
print(f"  pred_real_global  range: [{pred_real_global.min():.2f}, {pred_real_global.max():.2f}]")
print(f"  label_real_global range: [{label_real_global.min():.2f}, {label_real_global.max():.2f}]")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  METRIC: Mean Absolute Error (MAE) in Physical, Global Frame
# ══════════════════════════════════════════════════════════════════════════════

errors = (pred_real_global - label_real_global).abs()

mae_per_dim = errors.mean(dim=0)
mae_overall = errors.mean().item()

# Use the original global frame label names for clarity
GLOBAL_LABEL_NAMES = ["Δx", "Δy", "Δz", "Δyaw"]

print("=" * 55)
print("PHYSICAL MAE (TEST SET, GLOBAL FRAME)")
print("=" * 55)
for name, mae in zip(GLOBAL_LABEL_NAMES, mae_per_dim.tolist()):
    unit = "metres" if name != "Δyaw" else "radians"
    status = "✅" if (mae < 0.5 and unit == "metres") or (mae < 0.1 and unit == "radians") else "⚠️"
    print(f"  {status}  {name:<6}  MAE = {mae:.4f} {unit}")
print(f"\n  Overall MAE: {mae_overall:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  METRIC 1: Formation Precision (Euclidean 3D Position Error, Global Frame)
# ══════════════════════════════════════════════════════════════════════════════

# 3D Euclidean error in position (ignoring yaw)
pos_errors = (pred_real_global[:, :3] - label_real_global[:, :3])
euclidean_err = torch.norm(pos_errors, dim=1).numpy()

# Percentiles
p50 = np.percentile(euclidean_err, 50)
p90 = np.percentile(euclidean_err, 90)
p99 = np.percentile(euclidean_err, 99)
pct_under_0_5 = (euclidean_err < 0.5).mean() * 100

print("=" * 55)
print("FORMATION PRECISION (3D Euclidean Error, Global Frame)")
print("=" * 55)
print(f"  Median (p50)       : {p50:.4f} m")
print(f"  90th percentile    : {p90:.4f} m")
print(f"  99th percentile    : {p99:.4f} m")
print(f"  % within 0.5m      : {pct_under_0_5:.1f}%")
print()
result = "✅ PASS" if p90 < 0.5 else "⚠️  NEEDS IMPROVEMENT"
print(f"  Formation target (p90 < 0.5m): {result}")

# ── Distribution plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(euclidean_err, bins=150, color='#4a90d9', edgecolor='none', alpha=0.85,
        density=True, label='Error distribution')
ax.axvline(0.5, color='#e74c3c', ls='--', lw=2, label='Target: 0.5m')
ax.axvline(p50, color='#27ae60', ls='--', lw=1.5, label=f'Median: {p50:.3f}m')
ax.axvline(p90, color='#f39c12', ls='--', lw=1.5, label=f'p90: {p90:.3f}m')
ax.set_xlabel('3D Euclidean Error (metres)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Formation Precision — Position Error Distribution (Global Frame)', fontweight='bold', fontsize=13)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.set_xlim(0, min(3.0, np.percentile(euclidean_err, 99.5)))
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'formation_precision.png', dpi=150)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  VISUALIZATION: Predicted vs Actual Setpoints (Global Frame)
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Predicted vs Actual Setpoints (Test Set — Physical Global Frame)',
             fontweight='bold', fontsize=14)

colors = ['#4a90d9', '#e74c3c', '#27ae60', '#8e44ad']
units  = ['metres', 'metres', 'metres', 'radians']
GLOBAL_LABEL_NAMES = ["Δx", "Δy", "Δz", "Δyaw"]


# Subsample for plotting
n_plot = min(5000, pred_real_global.shape[0])
idx = np.random.choice(pred_real_global.shape[0], n_plot, replace=False)

for i, (ax, name, color, unit) in enumerate(zip(axes, GLOBAL_LABEL_NAMES, colors, units)):
    actual = label_real_global[idx, i].numpy()
    predicted = pred_real_global[idx, i].numpy()

    ax.scatter(actual, predicted, alpha=0.15, s=3, color=color)

    lims = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]
    ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.5, label='Perfect')

    r = np.corrcoef(actual, predicted)[0, 1]

    ax.set_xlabel(f'Actual ({unit})')
    ax.set_ylabel(f'Predicted ({unit})')
    ax.set_title(f'{name}  (R={r:.4f})', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pred_vs_actual_global.png', dpi=150)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  VISUALIZATION: Per-Component Error Distribution (Global Frame)
# ══════════════════════════════════════════════════════════════════════════════

residuals = (pred_real_global - label_real_global).numpy()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Prediction Residuals (Pred - Actual) in Global Frame — Should be Centered at 0',
             fontweight='bold', fontsize=13)

GLOBAL_LABEL_NAMES = ["Δx", "Δy", "Δz", "Δyaw"]

for i, (ax, name, color) in enumerate(zip(axes, GLOBAL_LABEL_NAMES, colors)):
    res = residuals[:, i]
    ax.hist(res, bins=100, color=color, edgecolor='none', alpha=0.85, density=True)
    ax.axvline(0, color='black', ls='--', lw=1)
    ax.axvline(res.mean(), color='red', ls='-', lw=1.5, label=f'μ={res.mean():.4f}')
    ax.set_title(f'{name}  (σ={res.std():.4f})', fontweight='bold', fontsize=10)
    ax.set_xlabel('Residual')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'residual_distributions_global.png', dpi=150)
plt.show()


### Metric 2: Scalability — Zero-Shot Generalization

A fundamental property of GNNs is **size invariance**: the same learned weights
process any number of nodes. Our SetpointGATv2 achieves this because:

1. **Node-level operations**: Each GATv2 layer computes per-node updates. The weight
   matrices are shared across all nodes — they don't depend on `N`.

2. **Attention aggregation**: GATv2's attention mechanism averages over whatever
   number of neighbors exist. A drone with 3 neighbors or 30 neighbors uses
   the same attention weights.

3. **No global pooling**: Our model never computes a graph-level embedding
   (no `global_mean_pool`). Every operation is strictly local → scalable.

**Trained on 10-20 drones → Deployed on 100:**
- The GNN processes each drone identically
- More drones = more edges in the communication graph
- Each drone still only "sees" neighbors within 10m radius
- The K=3 hop receptive field covers 30m regardless of swarm size

> **To verify in simulation:** Create a new dataset with 100 drones, run inference
> with the trained model, and measure formation precision. No retraining needed.

In [ ]:
# ── Zero-Shot Scalability Test (Synthetic) ─────────────────────────────────────
#
# We simulate a 100-drone graph by constructing a synthetic batch.
# This verifies that the model can process LARGER graphs than it trained on.
# NOTE: This tests computational compatibility, not formation quality.
# True quality requires sim evaluation.

from torch_geometric.data import Data as PyGData

N_SCALABILITY = 100  # drones
torch.manual_seed(42)

# Synthetic node features (random, within normalized range)
x_synth = torch.randn(N_SCALABILITY, len(INPUT_NAMES))
# Fix formation one-hot (all triangle)
x_synth[:, 6:] = 0
x_synth[:, 8] = 1  # form_tri

# Synthetic edges: random communication graph (avg ~8 neighbors per drone)
edges = []
for i in range(N_SCALABILITY):
    n_neighbors = np.random.randint(4, 12)
    neighbors = np.random.choice(
        [j for j in range(N_SCALABILITY) if j != i],
        size=min(n_neighbors, N_SCALABILITY - 1),
        replace=False
    )
    for j in neighbors:
        edges.append([i, j])
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_attr  = torch.randn(edge_index.shape[1], len(EDGE_NAMES))

# Run inference
synth_graph = PyGData(x=x_synth, edge_index=edge_index, edge_attr=edge_attr).to(DEVICE)

best_model.eval()
with torch.no_grad():
    t0 = time.time()
    synth_pred = best_model(synth_graph.x, synth_graph.edge_index, synth_graph.edge_attr)
    infer_time = (time.time() - t0) * 1000  # ms

synth_setpoints = best_normalizer.denormalize_target(synth_pred.cpu())

print("=" * 55)
print(f"SCALABILITY TEST — {N_SCALABILITY} Drones (trained on 10-20)")
print("=" * 55)
print(f"  Input shape    : [{N_SCALABILITY}, {len(INPUT_NAMES)}]")
print(f"  Edges          : {edge_index.shape[1]}")
print(f"  Output shape   : {tuple(synth_pred.shape)}")
print(f"  Inference time : {infer_time:.1f} ms")
print(f"  Hz (100 drones): {1000/infer_time:.0f} Hz")
print(f"\n  ✅ Model processes {N_SCALABILITY} drones with zero code changes")
print(f"  ⚠️  Quality must be validated in simulation (synthetic data used here)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  EVALUATION SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

test_mse = evaluate(best_model, test_loader, criterion, DEVICE)
GLOBAL_LABEL_NAMES = ["Δx", "Δy", "Δz", "Δyaw"]


print("\n" + "=" * 65)
print("FINAL EVALUATION SUMMARY")
print("=" * 65)
print(f"\n  Model          : SetpointGATv2 ({total_params:,} params)")
print(f"  Best epoch     : {ckpt['epoch']}")
print(f"  Test MSE (norm): {test_mse:.6f} (in local frame space)")
print(f"\n  Physical MAE (in global frame):")
for name, mae in zip(GLOBAL_LABEL_NAMES, mae_per_dim.tolist()):
    unit = 'm' if name != 'Δyaw' else 'rad'
    print(f"    {name:<6} = {mae:.4f} {unit}")
print(f"\n  Formation Precision (Global Frame):")
print(f"    Median 3D error : {p50:.4f} m")
print(f"    p90 3D error    : {p90:.4f} m")
print(f"    Within 0.5m     : {pct_under_0_5:.1f}%")
print(f"\n  Scalability: {N_SCALABILITY}-drone inference in {infer_time:.1f}ms")
print(f"\n  Checkpoint: {BEST_MODEL_PATH}")


---
## 6. DAgger Framework — Dataset Aggregation

### What is DAgger?

Standard imitation learning (behavioral cloning) trains on expert demonstrations only.
The problem: when the student (GNN) makes a small mistake, it enters a state the expert
never visited. The error compounds — this is called **distribution shift**.

**DAgger** fixes this by iteratively:
1. Rolling out the **student** policy in simulation
2. Querying the **teacher** (expert PID) for what it *would have done* in those states
3. Appending these new (state, expert_label) pairs to the training dataset
4. Retraining the student on the expanded dataset

```
  Iteration 0:  Expert flies → collect D₀ → train GNN₀
  Iteration 1:  GNN₀ flies  → expert labels GNN₀'s states → D₁ = D₀ ∪ new
                train GNN₁ on D₁
  Iteration 2:  GNN₁ flies  → expert labels → D₂ = D₁ ∪ new → train GNN₂
  ...
```

### Why DAgger matters for drone swarms

The expert PID flies with **perfect state knowledge**. The GNN only has **local
sensing + neighbor communication**. The GNN's trajectory will inevitably diverge
from the expert's. DAgger ensures the GNN learns to recover from its own mistakes
rather than only knowing the expert's comfortable states.

### DAgger's advantage over pure behavioral cloning

| Aspect | Behavioral Cloning | DAgger |
|--------|-------------------|--------|
| Training data | Expert states only | Expert + student states |
| Distribution shift | Compounds errors | Self-correcting |
| Data efficiency | Needs huge expert dataset | Grows iteratively |
| Recovery | Never learns to recover | Learns from own mistakes |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DAgger — Code Skeleton
#
#  This is a FRAMEWORK to be connected to PyFlyt simulation.
#  The core logic is complete; the sim interface stubs are marked with TODO.
# ══════════════════════════════════════════════════════════════════════════════

def dagger_iteration(
    model: SetpointGATv2,
    normalizer: SetpointNormalizer,
    existing_dataset: list,       # list of PyG Data graphs from previous iterations
    n_episodes: int = 20,         # number of rollout episodes per DAgger round
    steps_per_episode: int = 200,
    beta: float = 0.5,            # mixing ratio: β=1 → all expert, β=0 → all student
    device: str = "cpu",
):
    """
    One DAgger iteration:
      1. Roll out the student (GNN) in simulation
      2. At each step, query the expert (PID) for the correct setpoint
      3. Append new (state, expert_setpoint) pairs to dataset

    Args:
        model:            Trained SetpointGATv2
        normalizer:       SetpointNormalizer for input/output transforms
        existing_dataset: Previous training data (list of Data objects)
        n_episodes:       Number of simulation episodes
        steps_per_episode: Steps per episode
        beta:             Mixing coefficient (β×expert + (1-β)×student)
                          Start at 0.5, decay toward 0 over iterations
        device:           Torch device

    Returns:
        expanded_dataset: existing_dataset + new DAgger observations
    """
    model.eval()
    new_graphs = []

    for episode in range(n_episodes):
        # ── TODO: Initialize PyFlyt simulation ─────────────────────────────
        # env = PyFlytSwarmEnv(
        #     num_drones=np.random.randint(10, 21),
        #     formation=random.choice(["a", "rectangle", "triangle"]),
        # )
        # obs = env.reset()

        for step in range(steps_per_episode):
            # ── Step 1: Build graph from current sim state ─────────────────
            # TODO: Construct graph from env.get_drone_states()
            # raw_graph = build_graph_from_observation(obs)
            # norm_graph = normalizer.transform_graph(raw_graph)

            # ── Step 2: Student predicts setpoint ──────────────────────────
            # with torch.no_grad():
            #     student_pred_norm = model(
            #         norm_graph.x.to(device),
            #         norm_graph.edge_index.to(device),
            #         norm_graph.edge_attr.to(device),
            #     )
            #     student_setpoint = normalizer.denormalize_target(student_pred_norm.cpu())

            # ── Step 3: Expert (PID) provides ground-truth setpoint ────────
            # expert_setpoint = env.get_expert_pid_setpoint()

            # ── Step 4: Mix policies (for actual drone control) ────────────
            # action = beta * expert_setpoint + (1 - beta) * student_setpoint

            # ── Step 5: Step the simulation with mixed action ──────────────
            # obs = env.step(action)

            # ── Step 6: Store (current_state, expert_label) pair ───────────
            # This is the KEY DAgger insight: we record the EXPERT's
            # label for the STUDENT's (or mixed) state.
            # new_data = Data(
            #     x          = raw_graph.x,
            #     target     = expert_setpoint,  # EXPERT label for THIS state
            #     edge_index = raw_graph.edge_index,
            #     edge_attr  = raw_graph.edge_attr,
            #     pos        = raw_graph.pos,
            #     ...
            # )
            # new_graphs.append(new_data)
            pass

    # ── Merge datasets ─────────────────────────────────────────────────────
    expanded_dataset = existing_dataset + new_graphs
    print(f"  DAgger: {len(existing_dataset)} → {len(expanded_dataset)} graphs "
          f"(+{len(new_graphs)} from {n_episodes} episodes)")

    return expanded_dataset


print("DAgger skeleton ready ✅")
print("Connect to PyFlyt simulation to run actual DAgger iterations.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Full DAgger Training Loop (Structure)
# ══════════════════════════════════════════════════════════════════════════════

def run_dagger(
    initial_model,
    normalizer,
    initial_train_data,   # list of raw Data graphs
    n_iterations: int = 5,
    episodes_per_iter: int = 20,
    retrain_epochs: int = 30,
    device: str = "cpu",
):
    """
    Full DAgger loop.

    Schedule:
      Iter 0: β=1.0  (pure expert rollouts — same as behavioral cloning)
      Iter 1: β=0.7  (mostly expert, student starts contributing)
      Iter 2: β=0.5  (equal mix)
      Iter 3: β=0.3  (mostly student)
      Iter 4: β=0.1  (nearly pure student)
    """
    model = initial_model
    dataset = initial_train_data

    for iteration in range(n_iterations):
        # β decay: linear from 1.0 to 0.1
        beta = max(0.1, 1.0 - iteration * 0.2)

        print(f"\n{'='*60}")
        print(f"DAgger Iteration {iteration} (β={beta:.1f})")
        print(f"{'='*60}")

        # ── Step 1: Collect new data with current policy ───────────────────
        dataset = dagger_iteration(
            model=model,
            normalizer=normalizer,
            existing_dataset=dataset,
            n_episodes=episodes_per_iter,
            beta=beta,
            device=device,
        )

        # ── Step 2: Re-fit normalizer on expanded dataset ──────────────────
        # NOTE: Only if the data distribution shifted significantly.
        # For stability, you may keep the original normalizer.

        # ── Step 3: Retrain model on expanded dataset ──────────────────────
        # expanded_loader = DataLoader(
        #     NormalizedDroneDataset(dataset, normalizer),
        #     batch_size=32, shuffle=True
        # )
        # for epoch in range(retrain_epochs):
        #     train_loss = train_one_epoch(model, expanded_loader, ...)

        # ── Step 4: Evaluate ───────────────────────────────────────────────
        # val_loss = evaluate(model, val_loader, criterion, device)
        # print(f"  After retraining: val_loss={val_loss:.6f}")

    return model, dataset


print("Full DAgger loop structure ready ✅")
print("\nDAgger schedule:")
for i in range(5):
    b = max(0.1, 1.0 - i * 0.2)
    print(f"  Iter {i}: β={b:.1f}  ({'expert-heavy' if b > 0.5 else 'student-heavy'})")

---
## Summary

### What we built

```
SetpointGATv2
  ├── 3 × GATv2Conv (4 heads, 64-dim, K=3 hops)
  ├── Skip connections + LayerNorm + ELU
  ├── MLP head: 64 → 32 → 4 (NO Sigmoid)
  └── Output: [Δx, Δy, Δz, Δyaw] normalized setpoints
```

### Training recipe

| Setting | Value |
|---------|-------|
| Optimizer | Adam (lr=1e-4, wd=1e-6) |
| Scheduler | ReduceLROnPlateau (patience=5, factor=0.5) |
| Grad clip | 1.0 |
| Batch size | 32 |
| Early stop | Patience=15 |
| Loss | MSE (normalized space) |

### Key files

| File | Purpose |
|------|---------|
| `checkpoints/best_setpoint_gatv2.pt` | Best model + normalizer + hparams |
| `results/training_curves.png` | Loss + LR curves |
| `results/formation_precision.png` | Euclidean error distribution |
| `results/pred_vs_actual.png` | Scatter plots per setpoint dim |
| `results/residual_distributions.png` | Error analysis |

### Inference pipeline
```python
model, normalizer, _ = load_gnn_model("checkpoints/best_setpoint_gatv2.pt")
pred_norm = model(x, edge_index, edge_attr)
setpoint_real = normalizer.denormalize_target(pred_norm)
# → feed setpoint_real to PID controller
```

### Next steps
1. Run this notebook → evaluate formation precision
2. If p90 > 0.5m → increase `hidden_dim` or `num_layers`
3. Connect DAgger to PyFlyt simulation
4. Deploy in closed-loop simulation test